<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# Build a Dashboard Application with Plotly Dash

In this lab, you will be building a Plotly Dash application for users to perform interactive visual analytics on SpaceX launch data in real-time.

This dashboard application contains input components such as a dropdown list and a range slider to interact with a pie chart and a scatter point chart. You will be guided to build this dashboard application via the following tasks:

*   TASK 1: Add a Launch Site Drop-down Input Component
*    TASK 2: Add a callback function to render __success-pie-chart__ based on selected site dropdown
*    TASK 3: Add a Range Slider to Select Payload
*    TASK 4: Add a callback function to render the __success-payload-scatter-chart__ scatter plot

After visual analysis using the dashboard, you should be able to obtain some insights to answer the following five questions:

*    Which site has the largest successful launches?
*    Which site has the highest launch success rate?
*    Which payload range(s) has the highest launch success rate?
*    Which payload range(s) has the lowest launch success rate?
*    Which F9 Booster version (__v1.0, v1.1, FT, B4, B5,__ etc.) has the highest launch success rate?

**Estimated time needed:** 90 minutes

## Downloading and Prepping Data

If you are using local jupyter lab, then  add these lines in your code:

In [1]:
!pip install jupyter-dash dash==2.9.3 --quiet

## Read the Data

Let's start with

* Importing necessary libraries
* Reading the data

In [2]:
# Import required libraries
import pandas as pd
import plotly.express as px
from jupyter_dash import JupyterDash
from dash import dcc, html
from dash.dependencies import Input, Output

In [3]:
# Read the SpaceX data into pandas dataframe
spacex_df=pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv')
spacex_df.head()

,Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0
2,2,3,CCAFS LC-40,0,525.0,F9 v1.0 B0005,v1.0
3,3,4,CCAFS LC-40,0,500.0,F9 v1.0 B0006,v1.0
4,4,5,CCAFS LC-40,0,677.0,F9 v1.0 B0007,v1.0


In [4]:
min_payload=spacex_df["Payload Mass (kg)"].min()
max_payload=spacex_df["Payload Mass (kg)"].max()

print(min_payload, max_payload)

0.0 9600.0


In [5]:
print("Columnas disponibles:", spacex_df.columns.tolist())
print("Launch Sites:", spacex_df['Launch Site'].unique())
print("Datos cargados correctamente ✅")

Columnas disponibles: ['Unnamed: 0', 'Flight Number', 'Launch Site', 'class', 'Payload Mass (kg)', 'Booster Version', 'Booster Version Category']
Launch Sites: ['CCAFS LC-40' 'VAFB SLC-4E' 'KSC LC-39A' 'CCAFS SLC-40']
Datos cargados correctamente ✅


## TASK 1: Add a Launch Site Drop-down Input Component

We have four different launch sites and we would like to first see which one has the largest success count. Then, we would like to select one specific site and check its detailed success rate (class=0 vs. class=1).

As such, we will need a dropdown menu to let us select different launch sites.

Find and complete a commented __dcc.Dropdown(id='site-dropdown',...)__ input with following attributes:

*    __id__ attribute with value __site-dropdown__
*    __options__ attribute is a list of dict-like option objects (with label and value attributes). You can set the label and value all to be the launch  site names in the spacex_df and you need to include the default All option. e.g.,

  `options=[{'label': 'All Sites', 'value': 'ALL'},{'label': 'site1', 'value': 'site1'}, ...]`
  
*    __value__ attribute with default dropdown value to be ALL meaning all sites are selected
*    __placeholder__ attribute to show a text description about this input area, such as Select a Launch Site here
*    __searchable__ attribute to be True so we can enter keywords to search launch sites

In [6]:
# Create a dash application
app = JupyterDash(__name__)

In [7]:
# Dropdown
site_dropdown = dcc.Dropdown(
    id='site-dropdown',
    options=[
        {'label': 'All Sites', 'value': 'ALL'},
        {'label': 'CCAFS LC-40',  'value': 'CCAFS LC-40'},
        {'label': 'VAFB SLC-4E',  'value': 'VAFB SLC-4E'},
        {'label': 'KSC LC-39A',   'value': 'KSC LC-39A'},
        {'label': 'CCAFS SLC-40', 'value': 'CCAFS SLC-40'}
    ],
    value='ALL',
    placeholder="Select a Launch Site here",
    searchable=True,
    clearable=False
)

## TASK 2: Add a callback function to render success-pie-chart based on selected site dropdown

The general idea of this callback function is to get the selected launch site from site-dropdown and render a pie chart visualizing launch success counts.

Dash callback function is a type of Python function which will be automatically called by Dash whenever receiving an input component updates, such as a click or dropdown selecting event.

Let's add a callback function in __spacex_dash_app.py__ including the following application logic:

*    Input is set to be the __site-dropdown__ dropdown, i.e., __Input(component_id='site-dropdown', component_property='value')__
*    Output to be the graph with id __success-pie-chart__, i.e., __Output(component_id='success-pie-chart', component_property='figure')__
*    A __If-Else__ statement to check if ALL sites were selected or just a specific launch site was selected

*    If ALL sites are selected, we will use all rows in the dataframe spacex_df to render and return a pie chart graph to show the total success launches (i.e., the total count of class column)
*    If a specific launch site is selected, you need to filter the dataframe spacex_df first in order to include the only data for the selected site. Then, render and return a pie chart graph to show the success (__class=1__) count and failed (__class=0__) count for the selected site.

In [8]:
success_pie_chart = html.Div(dcc.Graph(id='success-pie-chart'))

## TASK 3: Add a Range Slider to Select Payload

Next, we want to find if variable payload is correlated to mission outcome. From a dashboard point of view, we want to be able to easily select different payload range and see if we can identify some visual patterns.

Find and complete a commented __dcc.RangeSlider(id='payload-slider',...)__ input with the following attribute:

*    __id__ to be __payload-slider__
*    __min__ indicating the slider starting point, we set its value to be 0 (Kg)
*    __max__ indicating the slider ending point to, we set its value to be 10000 (Kg)
*    __step__ indicating the slider interval on the slider, we set its value to be 1000 (Kg)
*    __value__ indicating the current selected range, we could set it to be __min_payload__ and __max_payload__

In [9]:
app.layout = html.Div([
    html.H1("SpaceX Launch Records Dashboard",
            style={'textAlign': 'center', 'color': '#503D36', 'fontSize': 30}),

    html.Br(),
    site_dropdown,
    html.Br(),

    html.Div(dcc.Graph(id='success-pie-chart'), style={'marginBottom': '30px'}),

    html.P("Payload range (Kg):", style={'fontSize': '18px'}),
    dcc.RangeSlider(
        id='payload-slider',
        min=min_payload,
        max=max_payload,
        step=500,
        value=[min_payload, max_payload],
        marks={i: str(i) for i in range(0, 10001, 2000)}
    ),
    html.Br(),

    html.Div(dcc.Graph(id='success-payload-scatter-chart'))
])

## TASK 4: Add a callback function to render the __success-payload-scatter-chart__ scatter plot

Next, we want to plot a scatter plot with the x axis to be the payload and the y axis to be the launch outcome (i.e., __class__ column). As such, we can visually observe how payload may be correlated with mission outcomes for selected site(s).

In addition, we want to color-label the Booster version on each scatter point so that we may observe mission outcomes with different boosters.

Now, let's add a call function including the following application logic:

*    Input to be __[Input(component_id='site-dropdown', component_property='value'), Input(component_id="payload-slider", component_property="value")]__ Note that we have two input components, one to receive selected launch site and another to receive selected payload range
*    Output to be __Output(component_id='success-payload-scatter-chart', component_property='figure')__
*    A __If-Else__ statement to check if ALL sites were selected or just a specific launch site was selected
*    If ALL sites are selected, render a scatter plot to display all values for variable __Payload Mass (kg)__ and variable __class__.

In addition, the point color needs to be set to the booster version i.e., __color="Booster Version Category"__

*    If a specific launch site is selected, you need to filter the __spacex_df__ first, and render a scatter chart to show values __Payload Mass (kg)__ and __class__ for the selected site, and color-label the point using __Boosster Version Category__ likewise.

In [10]:
@app.callback(
    [Output('success-pie-chart', 'figure'),
     Output('success-payload-scatter-chart', 'figure')],
    [Input('site-dropdown', 'value'),
     Input('payload-slider', 'value')]
)
def update_dashboard(selected_site, payload_range):
    low, high = payload_range

    # Filtro base por payload (aplica siempre a ambos gráficos)
    filtered_df = spacex_df[
        (spacex_df['Payload Mass (kg)'] >= low) &
        (spacex_df['Payload Mass (kg)'] <= high)
    ].copy()

    # Si se selecciona un sitio específico, filtrar también por sitio
    if selected_site != 'ALL':
        filtered_df = filtered_df[filtered_df['Launch Site'] == selected_site]

    # Depuración: ver cuántas filas quedan
    print(f"Site: {selected_site}, Payload: {low}-{high} kg → {filtered_df.shape[0]} filas")

    # --- PIE CHART ---
    if selected_site == 'ALL':
        # Usar filtered_df para respetar el rango de payload
        pie_fig = px.pie(
            filtered_df,
            names='Launch Site',
            values='class',
            title='Total Successful Launches by Launch Site (payload filtered)'
        )
    else:
        # Para un sitio, mostrar éxito (1) vs fracaso (0)
        success_counts = filtered_df['class'].value_counts().reset_index()
        success_counts.columns = ['class', 'count']
        pie_fig = px.pie(
            success_counts,
            names='class',
            values='count',
            title=f'Success vs Failure - {selected_site}',
            color='class',
            color_discrete_map={0: 'red', 1: 'green'}
        )
        pie_fig.update_traces(textinfo='percent+label')

    # --- SCATTER PLOT ---
    if filtered_df.empty:
        # Si no hay datos, mostrar un gráfico vacío con mensaje claro
        scatter_fig = px.scatter(title="No data for selected filters")
        scatter_fig.add_annotation(
            text=f"No hay lanzamientos en el rango de payload {low}–{high} kg<br>para el sitio: {selected_site}",
            x=0.5, y=0.5, showarrow=False, font=dict(size=14, color="red")
        )
    else:
        scatter_fig = px.scatter(
            filtered_df,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version Category',
            hover_data=['Booster Version'],
            title=f'Payload vs Outcome - {selected_site if selected_site != "ALL" else "All Sites"}',
            labels={'class': 'Outcome (1=Success, 0=Failure)'}
        )
        scatter_fig.update_yaxes(tickvals=[0, 1], ticktext=['Failure', 'Success'])
        scatter_fig.update_layout(height=500)

    return pie_fig, scatter_fig

In [11]:
app.run_server(mode='inline', height=1000, debug=False)

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/

INFO:werkzeug: * Running on http://127.0.0.1:8050/ (Press CTRL+C to quit)
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 01:12:45] "GET /_alive_6bd47a95-055e-42b0-bf85-157d43e69d50 HTTP/1.1" 200 -


<IPython.core.display.Javascript object>

Later in the browser address bar use this

http://localhost:8090

### Finding Insights Visually

Now with the dashboard completed, you should be able to use it to analyze SpaceX launch data, and answer the following questions:

*    Which site has the largest successful launches?
*    Which site has the highest launch success rate?
*    Which payload range(s) has the highest launch success rate?
*    Which payload range(s) has the lowest launch success rate?
*    Which F9 Booster version (v1.0, v1.1, FT, B4, B5, etc.) has the highest launch success rate?

                                                        © IBM Corporation 2021. All rights reserved.